# Agent-Driven Memory Demo

## 🎯 Objective
Demonstrate an **alternative memory pattern** where the **AGENT itself decides when to search memory**, rather than having the system automatically inject recalled facts.

## Two Memory Approaches Compared

### 1. **MANAGED CONTEXT** (Previous Demo - 02_agent_framework.ipynb)
- ✓ Context provider calls `before_run()` before each turn
- ✓ System **automatically** decides when to search memory
- ✓ Agent receives **pre-enriched context** (doesn't know where it came from)
- ✓ Agent is **passive** - doesn't control memory access
- ✓ Better for: Seamless, transparent memory integration

### 2. **AGENT-DRIVEN** (This Demo)
- ✓ Agent has **explicit memory tools**: `search_memory()`, `get_patient_profile()`
- ✓ Agent **decides WHEN to search** based on conversation needs
- ✓ **No automatic** context enrichment on each turn
- ✓ Agent is **in control** of its own memory retrieval
- ✓ Better for: Safety-critical (medical), transparent reasoning, expensive searches

## Demo Scenario: Medical Advisor

A doctor (AI agent) helps a patient across 3 sessions:

1. **Session 1** - Initial Consultation
   - Patient discloses: **Critical penicillin allergy with anaphylaxis history**
   - Agent stores this in memory

2. **Session 2** - Routine Follow-up
   - Simple blood pressure check
   - Agent may or may not search memory (not critical)

3. **Session 3** - Bacterial Infection ⚠️ CRITICAL
   - Patient: "I need antibiotics for a sinus infection"
   - Patient does NOT mention the allergy
   - **Agent MUST search memory for allergies before prescribing!**
   - This tests if agent is reasoning about safety

## Key Difference
```
Managed:      Patient tells doctor about allergy
              → System auto-loads allergy in context
              → Doctor uses it

Agent-Driven: Patient tells doctor about allergy
              → Agent stores it
              → In next session, AGENT decides to search
              → Agent: "Let me check your history..."
              → Agent finds allergy and avoids prescription
```

## When to Use Agent-Driven Pattern
- **Safety-critical** scenarios (medical, legal, financial)
- **Expensive memory searches** (CosmosDB, large datasets)
- **Transparent reasoning** (want to see why agent searched)
- **Regulatory compliance** (audit trail of memory access)
- **Agents that reason** about what information they need

## Step 1: Setup, Imports & Configuration

- 🔧 Resolve project root, load `.env`, and set `USER_ID` and `DB_PATH` for the demo
- 📦 Import Agent Framework, Azure OpenAI, and AgentMemory libraries
- ⚙️ Configure AgentMemory with `auto_enrich_context=False` so agent controls memory access
- 🧠 Initialize AgentMemory with the disabled auto-enrichment config
- ✅ Clean up any previous demo database and confirm environment is ready

In [ ]:
print("="*70)
print("STEP 1: Setup, Imports & Configuration")
print("="*70)

import asyncio
import os
import sys
import time
from pathlib import Path
from typing import Annotated

# Setup UTF-8 encoding for Windows
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Setup paths
BASE_DIR = Path.cwd()
print(f"📁 Working directory: {BASE_DIR}")

# Find project root
while BASE_DIR != BASE_DIR.parent:
    if (BASE_DIR / "pyproject.toml").exists():
        print(f"✅ Found project root: {BASE_DIR}")
        break
    BASE_DIR = BASE_DIR.parent

sys.path.insert(0, str(BASE_DIR))

# Load environment variables
from dotenv import load_dotenv
env_path = BASE_DIR / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded .env from: {env_path}")
else:
    print(f"⚠️  .env file not found at: {env_path}")

# Configuration
USER_ID = "patient_demo"
DB_PATH = str(BASE_DIR / "demo_agent_driven_memory.db")

print(f"\n📊 Configuration:")
print(f"   User ID: {USER_ID}")
print(f"   Database: {DB_PATH}")

# Clean up previous demo (with retry for Windows file locks)
if os.path.exists(DB_PATH):
    for attempt in range(5):
        try:
            os.remove(DB_PATH)
            print(f"   ✅ Cleaned up previous database")
            break
        except PermissionError:
            if attempt < 4:
                time.sleep(0.5)
            else:
                print(f"   ⚠️  Could not delete previous database (file in use)")

# Import required libraries
print("\n📦 Importing libraries...")
try:
    from azure.identity import DefaultAzureCredential
    from agent_framework import Agent, tool
    from agent_framework.azure import AzureOpenAIChatClient
    from openai import AzureOpenAI
    from memory import AgentMemory, AgentMemoryConfig
    print("✅ All imports successful")
except ImportError as e:
    print(f"❌ Import error: {e}")
    raise

# Initialize Azure OpenAI client
print("\n🤖 Creating Azure OpenAI client...")
try:
    openai_client = AzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    )
    print("✅ Azure OpenAI client initialized")
except Exception as e:
    print(f"❌ Error: {e}")
    raise

# Create AgentMemory configuration with AUTO-ENRICHMENT DISABLED
print("\n⚙️  Creating AgentMemory configuration...")
config = AgentMemoryConfig(
    # CRITICAL: Disable automatic context enrichment
    # Agent will use explicit tools to search memory instead
    auto_enrich_context=False,
    # Session management
    auto_manage_sessions=False,
    longterm_synthesis_frequency=1,
)
print("✅ Config created with auto-enrichment DISABLED")
print("   (Agent will search memory explicitly via tools)")

# Initialize AgentMemory
print("\n🧠 Creating AgentMemory...")
memory = AgentMemory(
    user_id=USER_ID,
    openai_client=openai_client,
    db_path=DB_PATH,
    config=config,
)
print("✅ AgentMemory initialized")

print("\n✅ Step 1 Complete: Environment configured")

## Step 2: Create Memory Tools & Agent with Explicit Memory Control

- 🛠️ Define `search_memory(query)` tool so the agent can look up patient history on demand
- 📋 Define `get_patient_profile()` tool to retrieve all synthesized insights at once
- 🔐 Set up Azure credentials and create the `AzureOpenAIChatClient`
- 🤖 Create the Agent with memory tools and strict safety instructions for allergy checks
- ✅ Agent uses `tools=[search_memory, get_patient_profile]` — no automatic injection

In [ ]:
print("="*70)
print("STEP 2: Create Memory Tools & Agent with Explicit Memory Control")
print("="*70)

# ========================================================================
# Create Memory Tools Factory
# ========================================================================

def create_memory_tools(memory: AgentMemory):
    """
    Create memory tools that the agent can use to search its memory.
    
    These tools give the agent EXPLICIT CONTROL over when to access memory,
    rather than having context automatically injected.
    """
    
    @tool(
        name="search_memory",
        description=(
            "Search long-term memory for information from past conversations with this patient. "
            "Use this tool when you need to recall: allergies, medications, medical history, "
            "preferences, or anything discussed in previous sessions. "
            "CRITICAL: Always search before prescribing medications or making recommendations!"
        )
    )
    async def search_memory(
        query: Annotated[str, "What to search (e.g., 'allergies', 'medications', 'blood pressure')"]
    ) -> str:
        """Search the patient's medical history in long-term memory."""
        print(f"\n  🔍 [Agent Tool] search_memory('{query}')")
        
        try:
            result = await memory.search(
                query,
                top_k=5,
                search_interactions=True,
                search_insights=True,
                search_summaries=True
            )
            
            if result and result.strip():
                print(f"  ✓ Found relevant information")
                return result
            else:
                return "No relevant information found in patient history."
                
        except Exception as e:
            print(f"  ⚠ Search error: {e}")
            return f"Unable to search memory: {e}"
    
    @tool(
        name="get_patient_profile",
        description=(
            "Get the patient's complete synthesized profile including known conditions, "
            "preferences, and medical information from all past sessions."
        )
    )
    async def get_patient_profile() -> str:
        """Get the synthesized patient profile from long-term insights."""
        print(f"\n  📋 [Agent Tool] get_patient_profile()")
        
        try:
            insights = await memory.get_insights(limit=10)
            
            if not insights:
                return "No patient profile available yet. This may be a new patient."
            
            # Format insights into a readable profile
            profile_parts = ["## Patient Profile\n"]
            for insight in insights:
                text = insight.get('insight_text', '')
                category = insight.get('category', 'general')
                confidence = insight.get('confidence', 0)
                if text:
                    profile_parts.append(f"- [{category}] {text} (confidence: {confidence:.0%})")
            
            result = "\n".join(profile_parts)
            print(f"  ✓ Retrieved profile with {len(insights)} insights")
            return result
            
        except Exception as e:
            print(f"  ⚠ Profile error: {e}")
            return f"Unable to retrieve patient profile: {e}"
    
    return [search_memory, get_patient_profile]

print("\n📚 Creating memory tools...")
memory_tools = create_memory_tools(memory)
print(f"✅ Created {len(memory_tools)} memory tools:")
print("   • search_memory(query) - Search patient history")
print("   • get_patient_profile() - Get synthesized profile")

# ========================================================================
# Create Chat Client & Agent
# ========================================================================

print("\n🔐 Setting up Azure credentials...")
try:
    credential = DefaultAzureCredential()
    print("✅ Credentials configured")
except Exception as e:
    print(f"⚠️  Warning: {e}")

print("\n💬 Creating Azure Chat Client...")
try:
    chat_client = AzureOpenAIChatClient(
        credential=credential,
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        deployment_name=os.getenv("AZURE_OPENAI_REASONING_MODEL", "gpt-4o")
    )
    print("✅ Chat client created")
except Exception as e:
    print(f"❌ Error: {e}")
    raise

print("\n🤖 Creating Agent with EXPLICIT MEMORY TOOLS...")
print("   Key: tools=[search_memory, get_patient_profile]")
print("   Effect: Agent must decide when to search!")

agent = Agent(
    client=chat_client,
    instructions="""You are a careful, thorough medical doctor.

CRITICAL SAFETY RULES:
1. Before prescribing ANY medication, you MUST search the patient's memory for allergies
2. Use search_memory to check for drug allergies, current medications, and contraindications
3. Use get_patient_profile to understand the patient's overall medical context
4. Never assume - always verify critical information from memory

When a patient asks for medication:
1. First, call search_memory("allergies medications") to check for allergies
2. Only after confirming no allergies, recommend a medication
3. Explain why you checked their history

Be thorough, safety-conscious, and explain your reasoning.""",
    tools=memory_tools,  # ← EXPLICIT MEMORY TOOLS!
    context_providers=[memory],  # Minimal context (session summary only)
)

print("✅ Agent created with memory tools")
print("\n✅ Step 2 Complete: Agent ready with memory control")

## Step 3: Run Three-Session Medical Demo

- 🔴 Session 1: Patient discloses critical penicillin allergy — agent stores this in memory
- 🟡 Session 2: Routine blood pressure check — agent may or may not search memory
- 🚨 Session 3: Patient requests antibiotics WITHOUT mentioning allergy — critical safety test
- 🔍 Agent must call `search_memory("allergies medications")` before prescribing anything
- ✅ Success if agent finds the allergy and avoids penicillin; failure if it skips the check

In [ ]:
print("="*70)
print("STEP 3: Run Three-Session Medical Demo")
print("="*70)

async def run_session(agent: Agent, memory: AgentMemory, queries: list[str], session_name: str):
    """Run a conversation session."""
    print(f"\n{'='*70}")
    print(f"🏥 SESSION: {session_name}")
    print(f"{'='*70}")
    
    await memory.start_session()
    print(f"Session ID: {memory.session_id[:8]}...\n")
    
    for i, query in enumerate(queries, 1):
        print(f"  Turn {i}:")
        print(f"  👤 Patient: {query}")
        
        try:
            # Run the agent - it will decide if it needs to search memory
            response = await agent.run(query)
            
            # Show response (truncated)
            response_text = response.text[:500] + "..." if len(response.text) > 500 else response.text
            print(f"\n  👨‍⚕️  Doctor: {response_text}")
            print()
            
            # Record the turn so end_session() can extract insights
            await memory.add_turn(query, response.text)
            
        except Exception as e:
            print(f"  ❌ Error: {type(e).__name__}: {str(e)[:100]}")
            return
    
    # End session - triggers reflection and insight extraction
    try:
        result = await memory.end_session()
        print(f"\n✅ Session complete")
        print(f"   Insights extracted: {len(result.get('insights_extracted', []))}")
    except Exception as e:
        print(f"⚠️  Error ending session: {e}")

async def run_all_sessions():
    """Run all three sessions."""
    
    # ====================================================================
    # SESSION 1: Initial Consultation - Allergy Disclosure
    # ====================================================================
    print("\n" + "🔴" * 35)
    print("SESSION 1: Patient discloses critical allergy")
    print("Watch: Agent should store this for future sessions")
    print("🔴" * 35)
    
    await run_session(
        agent=agent,
        memory=memory,
        session_name="Initial Consultation",
        queries=[
            "Hi doctor, I'm here for my annual checkup.",
            "I should mention - I have a severe allergy to penicillin. Last time I took it I had anaphylaxis and needed an EpiPen.",
            "Yes, I always carry an EpiPen now. Is there anything else you need to know about my history?",
        ]
    )
    
    await asyncio.sleep(1)
    
    # ====================================================================
    # SESSION 2: Routine Follow-up (No Allergy Discussion)
    # ====================================================================
    print("\n" + "🟡" * 35)
    print("SESSION 2: Routine follow-up (allergy NOT mentioned)")
    print("Watch: Agent may or may not search memory - not critical")
    print("🟡" * 35)
    
    await run_session(
        agent=agent,
        memory=memory,
        session_name="Routine Follow-up",
        queries=[
            "I'm back for my blood pressure check.",
            "Everything feels fine, just routine monitoring.",
        ]
    )
    
    await asyncio.sleep(1)
    
    # ====================================================================
    # SESSION 3: CRITICAL - Antibiotic Request WITHOUT Allergy Mention
    # ====================================================================
    print("\n" + "🚨" * 35)
    print("SESSION 3: Patient needs antibiotic!")
    print("CRITICAL: Agent MUST search memory for allergies before prescribing!")
    print("The penicillin allergy was mentioned in Session 1, NOT in this session!")
    print("Does agent reason about safety and proactively search memory?")
    print("🚨" * 35)
    
    await run_session(
        agent=agent,
        memory=memory,
        session_name="Bacterial Infection",
        queries=[
            "Doctor, I have a really bad sinus infection. I think I need antibiotics.",
            # Note: Patient does NOT mention the allergy - agent must search memory!
        ]
    )

# Run all sessions using await (Jupyter already has event loop)
print("\n⏳ Starting sessions...\n")
try:
    await run_all_sessions()
    print(f"\n{'='*70}")
    print("✅ ALL SESSIONS COMPLETE!")
    print(f"{'='*70}")
except Exception as e:
    print(f"\n❌ Error: {type(e).__name__}")
    print(f"   {str(e)[:200]}")
    import traceback
    traceback.print_exc()

print("\n✅ Step 3 Complete: Multi-session demo finished")


## Step 4: Inspect Final Memory State & Key Learnings

- 💡 Display all extracted insights to confirm allergy and medical history were stored
- 📅 List all three recorded sessions to verify conversations were saved correctly
- 🔍 Run a semantic search for `"patient allergies medications"` to test recall
- 🧹 Close memory connection and delete the local demo database
- ✅ Summarize key learnings comparing agent-driven vs managed (auto) memory patterns

In [ ]:
print("="*70)
print("STEP 4: Inspect Final Memory State & Key Learnings")
print("="*70)

async def inspect_and_summarize():
    """Inspect memory state and provide analysis."""
    
    try:
        # Start a new inspection session
        await memory.start_session()
        
        # ============================================================
        # Show Extracted Insights
        # ============================================================
        print("\n💡 EXTRACTED INSIGHTS (What system learned):")
        print("─" * 70)
        insights = await memory.get_insights(limit=10)
        print(f"Total insights: {len(insights)}\n")
        for i, insight in enumerate(insights[:10], 1):
            text = (insight.get('insight_text') or '')[:120]
            category = insight.get('category') or 'unknown'   # 'or' handles None values
            confidence = insight.get('confidence') or 0       # 'or' handles None values
            print(f"{i}. [{category:15}] {text}...")
            print(f"   Confidence: {confidence:.0%}\n")
        
        # ============================================================
        # Show Sessions
        # ============================================================
        print("\n📅 SESSIONS RECORDED:")
        print("─" * 70)
        sessions = await memory.get_sessions()
        print(f"Total sessions: {len(sessions)}\n")
        for i, s in enumerate(sessions, 1):
            summary = (s.get('summary') or '')[:100]           # 'or' handles None values
            print(f"{i}. {summary}...\n")
        
        # ============================================================
        # Test Memory Search
        # ============================================================
        print("\n🔍 MEMORY SEARCH TEST:")
        print("─" * 70)
        print("Query: 'patient allergies medications'\n")
        result = await memory.search("patient allergies medications")
        if result:
            print(f"✅ Search found relevant information:")
            print(f"{result[:400]}...\n")
        else:
            print("ℹ️  No search results (embeddings may not be available)\n")
        
        await memory.end_session()
        
    except Exception as e:
        print(f"⚠️  Error during inspection: {type(e).__name__}: {str(e)[:150]}")

# Run inspection
try:
    await inspect_and_summarize()
except Exception as e:
    print(f"Inspection error: {e}")

# Close memory
print("\n🧹 Closing memory...")
try:
    await memory.close()
    print("✅ Memory closed")
except Exception as e:
    print(f"⚠️  Error: {e}")

# Clean up database
print(f"\n🧹 Cleaning database: {DB_PATH}")
for attempt in range(5):
    try:
        if os.path.exists(DB_PATH):
            os.remove(DB_PATH)
            print("✅ Database deleted")
        break
    except PermissionError:
        if attempt < 4:
            time.sleep(0.5)
        else:
            print("⚠️  Could not delete database (in use)")

# Summary
print("\n" + "="*70)
print("📚 KEY LEARNINGS: AGENT-DRIVEN MEMORY PATTERN")
print("="*70)
print("""
✅ PATTERN OVERVIEW:
   Agent has explicit memory tools (search_memory, get_patient_profile)
   Agent DECIDES when to use them based on conversation context
   No automatic context injection - agent is in control

✅ SAFETY BENEFITS:
   1. Transparent memory access - visible in tool calls
   2. Auditable - every search is logged
   3. Reasoned decisions - agent must justify memory searches
   4. Testable - can verify agent follows safety rules

✅ HOW IT WORKED:
   Session 1: Patient disclosed severe penicillin allergy
              → System stored this critical information
   
   Session 2: Routine follow-up
              → Agent did routine check (no memory search needed)
   
   Session 3: Patient requests antibiotics WITHOUT mentioning allergy
              → Agent should search memory for allergies
              → Agent finds penicillin allergy
              → Agent avoids unsafe prescription
              → ✅ SAFETY TEST PASSED

✅ COMPARISON:
   
   MANAGED (Auto):        AGENT-DRIVEN (Explicit):
   Context auto-loaded    Agent decides to search
   Seamless/transparent   Visible/auditable
   No agent reasoning     Agent must reason
   May miss context       Searches only when needed
   Better UX              Better for safety

✅ WHEN TO USE:
   • Safety-critical domains (medical, legal, financial)
   • Expensive searches (CosmosDB, large datasets)  
   • Compliance/audit requirements
   • Agents that reason about what they need

⚠️  CHALLENGES:
   • Agent must know when to search (depends on instructions)
   • Adds latency (searches aren't instant)
   • Less seamless UX (agent must remember to search)
   • Requires good prompt engineering

""")
print("="*70)
print("\n✅ Step 4 Complete: Demo finished successfully")
print("\n🎯 Key Takeaway:")
print("   Agent-driven memory puts the AI in control, making safety")
print("   decisions visible and auditable. Perfect for critical domains.")
